<a href="https://colab.research.google.com/github/JustinRSK/2025_ML_EES/blob/main/Final_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%% 1. modules
import pandas as pd, numpy as np, matplotlib.pyplot as plt, rasterio
from itertools import combinations
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                             confusion_matrix, classification_report)
from sklearn.inspection import permutation_importance

#%% 2. load data and define classes
# Original map has 6 classes. Final scheme has 3:
#  - Pelouses et prairies + Forets are MERGED ("Vegetation terrestre"):
#    Jeffries-Matusita separability = 0.43, far below the 1.7 threshold,
#    so they cannot be reliably distinguished at 30 m.
#  - Milieux construits + Plantations are EXCLUDED: f1 = 0.00 in testing,
#    not detectable at this resolution.
# dist_to_water is excluded from predictors: it was derived from the
# vegetation labels themselves (label leakage) and cannot be computed
# in the unlabelled reserves 8 and 9.

df = pd.read_csv('data/GC_training_table_2011.csv')
df['orig'] = df['class_numeric'].astype(int)
df['class'] = df['orig'].map({1:1, 2:2, 3:3, 6:3})   # 4,5 -> NaN = excluded
df = df.dropna(subset=['class']).copy()
df['class'] = df['class'].astype(int)

CLASS_NAMES = {1:'Eaux libres', 2:'Rivages et lieux humides', 3:'Vegetation terrestre'}
PRED = ['blue','green','red','nir','swir1','swir2','NDVI','NDMI','MNDWI','NBR',
        'elev_rel_lake','slope']

print('Pixels:', len(df), '| Predictors:', len(PRED), '| Reserves:', sorted(df.reserve_id.unique()))
print(df['class'].map(CLASS_NAMES).value_counts())